<a href="https://colab.research.google.com/github/VitorOliveira-Dev09/security-password/blob/main/Security_PassWord.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [51]:
import math
import string
import random
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

In [40]:
#Definindo dados brutos em variaveis, pois fica mais facil para o modelo entender
FEATURE_NAMES = [
    "comprimento",
    "tem_maiuscula",
    "tem_minuscula",
    "tem_numero",
    "tem_especial",
    "qtd_maiusculas",
    "qtd_numeros",
    "qtd_especiais",
    "chars_unicos",
    "entropia",
]

FEATURE_LABELS = {
    "comprimento":    "Senha muito curta (menos de 8 caracteres)",
    "tem_maiuscula":  "Sem letras maiúsculas",
    "tem_minuscula":  "Sem letras minúsculas",
    "tem_numero":     "Sem números",
    "tem_especial":   "Sem caracteres especiais (!@#...)",
    "qtd_maiusculas": "Poucas letras maiúsculas",
    "qtd_numeros":    "Poucos números",
    "qtd_especiais":  "Poucos caracteres especiais",
    "chars_unicos":   "Muitos caracteres repetidos",
    "entropia":       "Senha previsível / baixa entropia",
}



In [41]:
"""
Função Auxiliar para treinamento do modelo, medir o quanto imprevisível pode ser a senha
Usamos a formula de shannon, para a IA saber que quanto maior a impresivibilidade, maior a segurança
Acaba que temos uma relação de função de crescimento y = f(X).
"""
def calcular_entropia(senha: str) -> float:
    if not senha:
        return 0.0
    freq = {}
    for c in senha:
        freq[c] = freq.get(c, 0) + 1
    return -sum((v / len(senha)) * math.log2(v / len(senha)) for v in freq.values())


In [42]:
"""
Nessa função estamos extraindo a senha, em modo de vetor, para a ML entender de forma numérica oque se passa
Seria algo tipo
senha = "abc123"
vetor = [6(porque tem 6 caracteres),  0, 1, 1, 0,  0, 3, 0,  6,  2.58]
         ^                                                       ^     ^
       curta                                                   única  entropia baixa
"""
def extrair_features(senha: str) -> list:
    return [
        len(senha),
        int(any(c.isupper() for c in senha)),
        int(any(c.islower() for c in senha)),
        int(any(c.isdigit() for c in senha)),
        int(any(c in string.punctuation for c in senha)),
        sum(1 for c in senha if c.isupper()),
        sum(1 for c in senha if c.isdigit()),
        sum(1 for c in senha if c in string.punctuation),
        len(set(senha)),
        calcular_entropia(senha),
    ]

In [43]:
"""
Bem finalmente estamos treiando o modelo(ou tentando kkkkk)
Modelo de senha fraca para alimentar nosso modelo
lambda: cria uma lista de funções (cada lambda representa um tipo fraco de senha)
random.choice: sorteia uma das funções e da senha também
"""
def gerar_senha_fraca() -> str:
    opcoes = [
        lambda: random.choice(["123456", "password", "qwerty", "abc123"]),
        lambda: "".join(random.choices(string.ascii_lowercase, k=random.randint(4, 7))),
        lambda: "".join(random.choices(string.digits, k=random.randint(4, 8))),
        lambda: random.choice(["nome123", "data2024", "senha@1"]),
    ]
    return random.choice(opcoes)()

In [44]:
"""
Ao contrário da função acima agora geramos uma senha mega forte
Temos garantia de que temos um tipo de cada caracter, escolhemos outros aleatorios depois embaralhamos
"""
def gerar_senha_forte() -> str:
    tamanho = random.randint(10, 16)
    obrigatorios = [
        random.choice(string.ascii_uppercase),
        random.choice(string.ascii_lowercase),
        random.choice(string.digits),
        random.choice(string.punctuation),
    ]
    resto = random.choices(
        string.ascii_letters + string.digits + string.punctuation,
        k=tamanho - 4
    )
    senha = obrigatorios + resto
    random.shuffle(senha)
    return "".join(senha)

In [45]:
"""
Aparte que mais quebramos a cabeça, enteder matematicamente a matriz, vimos exemplos, mais cada sistema
tem suas particulariades, nada funciona no "crtl c" & "crtl v".
X e a matriz de features
y o vetor
Quando estavamos testando, analisamos que quando so pelo fato a senha ter maisculas o sistema ja garantia que ela era forte
usamos o StandScaler para padrozinar todos esses features
FINALMENTE:
O RandomForest está criando 100 tipos de arvores diferentes com decisões distintas uma das outras
Analisamos que por baixo dos panos funciona igual a uma votação parlamentar pública, eles analisa e vota, no final a maioria ganha
"""
def treinar_modelo() -> tuple:
    print("Treinando modelo ML... ", end="")

    X, y = [], []
    for _ in range(1500):
        s = gerar_senha_fraca()
        X.append(extrair_features(s))
        y.append(0)  # fraca

    for _ in range(1500):
        s = gerar_senha_forte()
        X.append(extrair_features(s))
        y.append(1)  # forte

    X = np.array(X)
    y = np.array(y)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    modelo = RandomForestClassifier(
        n_estimators=100,
        max_depth=8,
        random_state=42
    )
    modelo.fit(X_scaled, y)

    print(f"Pronto! Acurácia estimada: {modelo.score(X_scaled, y) * 100:.1f}%\n")
    return modelo, scaler


Para entender como funciona o RandomForest assistimos dois videos muitos bons <br>
https://youtu.be/MMpeBCWBjhc?si=KLJ_vkJS5cy_Wjo0

https://youtu.be/W-fCDIjxIQY?si=bSYxoQ96LgcVp7Nb


No video ele fala como funciona as as Arvóres de Decisão

-VITOR: Em meu tcc do técnico fiz algo parecido(seria mais uma gambiarra), para indicar ecopontos proxímo do usuário, utilizava LAT e LONG para o chatbot indicar aonde hávia o ecoponto

In [46]:
"""
Algoritmo Inteligente = AI ?

Algortimo que aplica regras que nos definimos, para um sistema "simples", somente isso bastava, mais segurança vai além disso...
"""
def analyze_heuristic_password(senha: str, dados_cadastro: dict, leaked_passwords: list) -> tuple:
    problemas = []
    pontuacao = 0

    checks = [
        (len(senha) >= 8,                              "Menos de 8 caracteres"),
        (any(c.isdigit() for c in senha),              "Sem números"),
        (any(c.islower() for c in senha),              "Sem letras minúsculas"),
        (any(c.isupper() for c in senha),              "Sem letras maiúsculas"),
        (any(c in string.punctuation for c in senha),  "Sem caracteres especiais"),
    ]

    for passou, desc in checks:
        if passou:
            pontuacao += 1
        else:
            problemas.append(desc)

    if senha in leaked_passwords:
        problemas.append("Encontrada em lista de senhas vazadas")
    else:
        pontuacao += 1

    if dados_cadastro:
        partes = (
            dados_cadastro['name'].lower().split()
            + dados_cadastro['email'].lower().replace('.', ' ').replace('@', ' ').split()
            + dados_cadastro['dt_birth'].split('/')
            + dados_cadastro['moment_important'].lower().split()
        )
        for palavra in set(partes):
            if len(palavra) > 2 and palavra in senha.lower():
                problemas.append(f"Contém dado pessoal: '{palavra}'")
                pontuacao -= 2

    return pontuacao, problemas, extrair_features(senha)

In [47]:
"""
Função para retorno da resposta da ML, somente apartir da segunda tentativa acontece algo

historico = dicionario que representa uma tentativa

X = np.array([t['features'] for t in historico]) = funciona como uma lista que empilha em uma matriz, e transforma a lista em NumPY para o sklearn

"""
def veredicto_ml(historico: list, modelo, scaler) -> None:
    """
    Compara os vetores de features entre tentativas e usa o modelo
    para identificar o padrão de fraqueza recorrente do usuário.
    """
    print("\n── Veredicto do Modelo Security PassWord - ML ──")

    # Predição de força para cada tentativa
    X = np.array([t['features'] for t in historico])
    X_scaled = scaler.transform(X)
    probabilidades = modelo.predict_proba(X_scaled)[:, 1]  # prob. de ser forte

    print("Histórico de força detectada pelo modelo:")
    for i, prob in enumerate(probabilidades, 1):
        barra = "█" * int(prob * 20) + "░" * (20 - int(prob * 20)) # Barra de progresso
        print(f"  Tentativa {i}: [{barra}] {prob * 100:.1f}%")

    # Detecta features consistentemente fracas
    # Para cada feature, verifica se melhorou, piorou ou ficou igual
    nomes = FEATURE_NAMES
    primeira = np.array(historico[0]['features'])
    ultima   = np.array(historico[-1]['features'])
    delta    = ultima - primeira

    # Features binárias que nunca foram atendidas em NENHUMA tentativa
    features_sempre_zero = []
    for i, nome in enumerate(nomes): # para cada "i" coletamos o valor em todas tentativas, porque vai que na segunda tentativa o usuario passou uma senha forte e na quarta não
        valores = [t['features'][i] for t in historico]
        if max(valores) == 0:
            features_sempre_zero.append(nome)

    # Features que pioraram entre a primeira e a última tentativa
    features_pioraram = [
        nomes[i] for i, d in enumerate(delta) if d < 0
    ]

    print("\nPadrão identificado:")

    if features_sempre_zero:
        print("  ❌ Você NUNCA usou:")
        for f in features_sempre_zero:
            print(f"     -> {FEATURE_LABELS[f]}")

    if features_pioraram:
        print("  📉 Piorou entre tentativas:")
        for f in features_pioraram:
            print(f"     -> {FEATURE_LABELS[f]}")

    # Tendência geral
    tendencia = probabilidades[-1] - probabilidades[0]
    if tendencia > 0.1:
        print("\n  ✅ Tendência: você está MELHORANDO a cada tentativa.")
    elif tendencia < -0.1:
        print("\n  ⚠️  Tendência: suas senhas estão ficando mais FRACAS.")
    else:
        print("\n  ➡️  Tendência: sem melhora significativa entre tentativas.")

    # Importância de features segundo o RandomForest
    importancias = modelo.feature_importances_
    top_features = np.argsort(importancias)[::-1][:3]
    print("\n  💡 O que mais pesa na avaliação do modelo:")
    for idx in top_features:
        print(f"     -> {FEATURE_LABELS[nomes[idx]]} "
              f"(importância: {importancias[idx] * 100:.1f}%)")

In [48]:
#Função inicial para capturar os dados do usuário
def register_user() -> dict:
    name = input("Digite seu Nome: ")
    email = input("Digite seu Email: ")
    dt_birth = input("Digite sua Data de Nascimento (DD/MM/AAAA): ")
    moment_important = input("Descreva um momento importante em sua vida: ")
    print("Cadastro concluído com sucesso!")
    return {
        "name": name,
        "email": email,
        "dt_birth": dt_birth,
        "moment_important": moment_important
    }

In [49]:
def read_leaked_passwords() -> list:
  """
  Lê senhas vazadas de um arquivo de dicionário que especifiquei e as retorna como uma lista.
  Trata o erro FileNotFoundError caso o dicionário não seja encontrado.
  """
  try:
    with open('/content/drive/MyDrive/Faculdade/Programação /milw0rm-dictionary.txt', 'r', encoding='utf-8') as arquivo:
        leaked_passwords = [line.strip() for line in arquivo if line.strip()]
    print(f"Arquivo carregado com sucesso, {len(leaked_passwords)} senhas vazadas para análise.")
    return leaked_passwords
  except FileNotFoundError:
    print("Error: O dicionário de senhas vazadas 'milw0rm-dictionary.txt' não existe em '/content/drive/MyDrive/Faculdade/Programação /milw0rm-dictionary.txt'. "
          "Certifique-se de que o arquivo existe no caminho especificado ou informe o caminho certo")
    return []
  except Exception as e:
    print(f"Ocorreu um erro ao ler o arquivo de senhas vazadas: {e}")
    return []


In [50]:
def main():
    print("=" * 52)
    print("  Security PassWord Fatec — ML Edition")
    print("=" * 52 + "\n")

    modelo, scaler   = treinar_modelo()
    dados_usuario    = register_user()
    leaked_passwords = read_leaked_passwords()

    historico: list[dict] = []

    while True:
        print("── Nova Tentativa ──")
        senha = input("Senha para analisar: ")
        if not senha:
            print("Senha vazia.\n")
            continue

        pontuacao, problemas, features = analyze_heuristic_password(
            senha, dados_usuario, leaked_passwords
        )

        # Resultado da tentativa atual
        print(f"\nProblemas encontrados:")
        if problemas:
            for p in problemas:
                print(f"  ✗ {p}")
        else:
            print("  ✓ Nenhum problema básico encontrado.")
        print(f"Pontuação heurística: {pontuacao}")

        historico.append({
            "pontuacao": pontuacao,
            "problemas": problemas,
            "features":  features,
        })

        # Veredicto ML ativa a partir da 2ª tentativa
        if len(historico) >= 2:
            veredicto_ml(historico, modelo, scaler)

        continuar = input("\nAnalisar outra senha? (s/n): ").lower()
        if continuar != 's':
            break

    print("\nObrigado por usar o Security PassWord Fatec!")


main()

  Security PassWord Fatec — ML Edition

Treinando modelo ML... Pronto! Acurácia estimada: 100.0%

Digite seu Nome: Vitor Oliveira
Digite seu Email: vitordev.inf@gmail.com
Digite sua Data de Nascimento (DD/MM/AAAA): 25042008
Descreva um momento importante em sua vida: VitorLindo
Cadastro concluído com sucesso!
Arquivo carregado com sucesso, 84198 senhas vazadas para análise.
── Nova Tentativa ──
Senha para analisar: 123

Problemas encontrados:
  ✗ Menos de 8 caracteres
  ✗ Sem letras minúsculas
  ✗ Sem letras maiúsculas
  ✗ Sem caracteres especiais
  ✗ Encontrada em lista de senhas vazadas
Pontuação heurística: 1

Analisar outra senha? (s/n): n

Obrigado por usar o Security PassWord Fatec!


NOSSA DEFESA: Obvio que utilizamos IA, principalmente para entender erros no codigo, sugestão de melhorias, resposta da ML bonitinha para o usuário, conceitos entre outros. Sabemos que fugimos um pouco do conceito de programação estruturada, trabalhamos com lista, dicionarios entre outros, mais a base continua a mesma

___________________________________

As vezes o arquivo de 8.000 senhas não salva no colab, não sabemos pq, se isso acontecer vamos deixar pronto para você poder usar





